In [ ]:
import math
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

def plot(title, label, x, result, expected, interval=None):
    if interval is not None:
        plt.axis([-10.1, 10.1, min(result), max(result)])
    plt.plot(x, expected, label=f'Ground truth')
    plt.plot(x, result, label=f'Sequre {label}')
    plt.title(title)
    plt.xlabel('x')
    plt.ylabel('y')
    plt.legend()
    plt.grid(True)
    plt.show()

def mae(result, expected):
    return np.mean(np.abs(np.array(result) - np.array(expected)))

def mxae(result, expected):
    return np.max(np.abs(np.array(result) - np.array(expected)))

def by_interval(df):
    for interval, group in df.groupby('Interval'):
        display(group)

In [ ]:
""" Core methods """

# Until Codon Jupyter is fixed: Read the data from files
show_plots = False

dump_folder = "dump"
dump_files = [
    "decor_trig",
    "decor",
    "fourier",
    "cheby",
    "taylor"
    ]
nbit_fs = [64]
intervals_count = 4
cps = [0, 1]
exclude = []

df_data = {
    'Method': [],
    'Interval': [],
    'MAE': [],
    'MXAE': [],
    'Runtime (avg)': [],
    'Runtime (std)': [],
    'Partitions count': [],
    'Truncations count': [],
    'Rounds': []
    }

df_waves = {
    'x': [],
    'Method': [],
    'Expected': [],
    'Result': []
    }

for cp in cps:
    df_data[f'Bytes sent CP{cp}'] = []
    df_data[f'Requests sent CP{cp}'] = []

for dump_file in dump_files:
    for nbit_f in nbit_fs:
        for i in range(intervals_count):
            for cp in cps:
                try:
                    with open(f"{dump_folder}/{dump_file}_{i}_{nbit_f}_CP{cp}.p", "rb") as f:
                        data = pickle.load(f)
                        x = data['x']
                        interval = f"({round(min(x), 2)}, {round(max(x), 2)})"
                        for k, v in data.items():
                            if not k.endswith('_result'):
                                continue

                            skip = False
                            for exclude_item in exclude:
                                if exclude_item in k:
                                    skip = True
                                    break
                            
                            if skip:
                                continue

                            k = k.replace('_result', '')
                            expected = data[f"{k}_expected"]

                            runtime_avg = round(data[f"{k}_time"][0], 5)
                            runtime_std = round(data[f"{k}_time"][1], 5)
                            bytes_sent = int(data[f"{k}_bytes_sent"][0])
                            send_requests = int(data[f"{k}_send_requests"][0])
                            partitions_count = int(data[f"{k}_partitions_count"][0])
                            truncations_count = int(data[f"{k}_truncations_count"][0])
                            rounds = int(data[f"{k}_rounds"][0])
                            
                            if cp == 1:
                                df_data['Method'].append(f"{k}_{nbit_f}")
                                df_data['Interval'].append(interval)
                                df_data['MAE'].append(mae(v, expected))
                                df_data['MXAE'].append(mxae(v, expected))

                                df_data['Runtime (avg)'].append(runtime_avg)
                                df_data['Runtime (std)'].append(runtime_std)
                                df_data['Partitions count'].append(partitions_count)
                                df_data['Truncations count'].append(truncations_count)
                                df_data['Rounds'].append(rounds)

                            df_data[f'Bytes sent CP{cp}'].append(bytes_sent)
                            df_data[f'Requests sent CP{cp}'].append(send_requests)

                            if cp == 1 and interval == "(-9.42, 9.42)":
                                df_waves['x'].append(x)
                                df_waves['Method'].append(f"{k}_{nbit_f}")
                                df_waves['Expected'].append(expected)
                                df_waves['Result'].append(v)

                            if show_plots and cp == 1:
                                plot(f"{k} {nbit_f} frac bits on {interval}", k, x, v, expected)
                except FileNotFoundError:
                    print(f"Could not find {dump_folder}/{dump_file}_{i}_{nbit_f}.p")

# Calculate total network overhead as the sum of bytes sent by both parties
if cps == [0, 1]:
    df_data[f'Bytes sent total'] = [
        df_data[f'Bytes sent CP0'][i] + df_data[f'Bytes sent CP1'][i] * 2
        for i in range(len(df_data['Method']))
    ]

df = pd.DataFrame(df_data)

In [ ]:
if df.empty:
    raise ValueError("DataFrame is empty")

display_methods = [
    "sin_",
    "cos_",
    "tan_",
    "cot",
    "exp",
    "sigmoid",
    "sinh",
    "cosh",
    "tanh",
    # "sqrt",
    # "log",
    # "mul_inv",
    "polynomial"
    ]

for method in display_methods:
    display(by_interval(df[df['Method'].str.contains(method)].sort_values(by='MAE')))

In [ ]:

# Filter methods containing both 'chebyshev' and 'decor' and compute averages
mask = df['Method'].str.contains('chebyshev_20') & df['Method'].str.contains('decor')
df_cheby_decor = df[mask].copy()

if df_cheby_decor.empty:
    print("No methods matching both 'chebyshev_20' and 'decor' found.")
else:
    numeric_cols = df_cheby_decor.select_dtypes(include=[np.number]).columns
    averages = df_cheby_decor[numeric_cols].mean().to_frame().T
    averages['Methods_count'] = len(df_cheby_decor)
    display(df_cheby_decor)   # rows included in the average
    print("Averaged numeric values for methods with both 'chebyshev_20' and 'decor':")
    display(averages)

In [ ]:
import re

def by_name(name, df):
    return df[df['Method'].str.startswith(name)]

def plot_accuracy_vs_perf(
        df, methods=["sin", "cos", "exp", "sigmoid"],
        metric="Runtime (avg)", intervals=["(-20.0, 20.0)"],
        approaches=["decor", "chebyshev", "fourier", "taylor"],
        markers=["o", "s", "^", "v"], colors=["tab:blue", "tab:orange", "tab:green", "tab:red"],
        remove_outliers=False, arangement="horizontal",
        print_legend=False, print_labels=False, force_labels=False,
        scale=None):
    
    if arangement == "horizontal":
        fig, axs = plt.subplots(1, len(methods), figsize=(5 * len(methods), 5))
    elif arangement == "vertical":
        fig, axs = plt.subplots(len(methods), 1, figsize=(5, 5 * len(methods)))
    elif arangement == "square":
        n = int(np.ceil(np.sqrt(len(methods))))
        fig, axs = plt.subplots(n, n, figsize=(5 * n, 5 * n))
        axs = axs.flatten()
    else:
        raise ValueError(f"Unknown arangement: {arangement}")
    for idx, method in enumerate(methods):
        ax = axs[idx]
        ax.grid(True)


        filtered_df = df[df['Method'].str.contains(f"{method}_")]
        filtered_df = filtered_df[
            ~filtered_df['Method'].str.contains('_decor') &
            (~filtered_df['Method'].str.contains('_naive') |
             filtered_df['Method'].str.contains('taylor_'))
        ]
        filtered_df['Method'] = filtered_df['Method'].str.replace(f"{method}_", "", regex=False)
        
        if remove_outliers:
            filtered_df = filtered_df[filtered_df['MAE'] < 0.5]
        
        # Convert MAE to accuracy
        filtered_df['MAE'] = filtered_df['MAE'].apply(lambda x: -math.log2(x) if x > 0 else 100)
        
        for interval, group in filtered_df.groupby('Interval'):
            if interval not in intervals:
                print(f"Skipping interval {interval}")
                continue
            for approach, marker, color in zip(approaches, markers, colors):
                for nbit_f in nbit_fs:
                    filtered_group = group[
                        group['Method'].str.startswith(approach) &
                        group['Method'].str.endswith(f"{nbit_f}")
                    ].sort_values(by=[metric, "MAE"])
                    if filtered_group.empty:
                        continue
                    if any(a in approach for a in ["chebyshev", "fourier", "taylor"]):
                        ax.plot(
                            filtered_group[metric], filtered_group["MAE"],
                            label=approach.capitalize(), 
                            linestyle='--' if nbit_f == 64 else '-', marker=marker, linewidth=2,
                            markersize=10,
                            color=color
                        )
                        if "chebyshev" in approach or force_labels:
                            # Add label to each Chebyshev point
                            label_idx = 0
                            for x, y, name in zip(filtered_group[metric], filtered_group["MAE"], filtered_group['Method']):
                                label_idx += 1
                                if label_idx % 2 == 1:
                                    continue
                                # Try to parse first number from the left in approach string
                                match = re.search(r'\d+', name)
                                label = match.group(0) if match else name
                                ax.text(x, y, label, fontsize=19, ha='right', va='bottom')
                    else:
                        ax.scatter(
                            filtered_group[metric], filtered_group["MAE"],
                            label=approach.capitalize(), marker=marker, s=150,
                            color=color
                        )
            if print_labels:
                ax.set_title(f"{method} at {interval}")
                ax.set_xlabel(metric)
                ax.set_ylabel("Accuracy: -log2(MAE)")
            ax.tick_params(axis='both', which='major', labelsize=19)
            ax.tick_params(axis='both', which='minor', labelsize=19)
        if idx == len(methods) - 1 and print_legend:
            ax.legend(loc='lower right')

    if scale is not None:
        # Format x-axis in millions and annotate xlabel with scale
        fmt = FuncFormatter(lambda x, pos: f"{x/scale:.1f}")
        axes = np.array(axs).flatten()
        for ax in axes:
            ax.xaxis.set_major_formatter(fmt)

    plt.tight_layout()
    plt.show()

In [ ]:
plot_accuracy_vs_perf(df, metric="Runtime (avg)", approaches=["decor", "chebyshev", "taylor"])
plot_accuracy_vs_perf(df, metric="Bytes sent total", approaches=["decor", "chebyshev", "taylor"], scale=1e6)

In [ ]:
# Plot sine, square wave, and sawtooth wave results and expected for degree, Chebyshev and Fourier

wave_types = ['sin', 'square', 'sawtooth']
degree = 50
approaches = ['chebyshev', 'fourier']
colors = ['tab:green', 'tab:orange', 'tab:blue']
print_labels = False

fig, axs = plt.subplots(len(wave_types), 1, figsize=(10, 4 * len(wave_types)), sharex=True)

for idx, wave in enumerate(wave_types):
    ax = axs[idx]
    mask = [
            (wave in m) and
            (f'_{degree}' in m)
            for m in df_waves['Method']
        ]
    if not any(mask):
        continue
    
    x = np.array(df_waves['x'])[mask][0]
    ground_truth = np.array(df_waves['Expected'])[mask][0]
    ax.plot(x, ground_truth, label=f'Ground truth', color=colors[0])
    for i, approach in enumerate(approaches):
        mask = [
            (wave in m) and
            (approach in m) and
            (f'_{degree}' in m)
            for m in df_waves['Method']
        ]
        if not any(mask):
            continue
        result = np.array(df_waves['Result'])[mask][0]
        x = np.array(df_waves['x'])[mask][0]
        ax.plot(
            x, result,
            label=f'{"Decor " if "fourier" in approach else ""}{approach.capitalize()}',
            linestyle='--', color=colors[i + 1])
    
    ax.tick_params(axis='both', which='major', labelsize=19)
    ax.tick_params(axis='both', which='minor', labelsize=19)
    if print_labels:
        ax.set_title(f'{wave.capitalize()} Wave (Degree {degree})')
        if idx == 0:
            ax.legend()


plt.xlabel('x')
plt.tight_layout()
plt.show()

In [ ]:
display(by_interval(df[df['Method'].str.contains("sawtooth")].sort_values(by='MAE')))

In [ ]:
plot_accuracy_vs_perf(
        df, methods=["sin", "square", "sawtooth"],
        metric="Runtime (avg)", intervals=["(-9.42, 9.42)"],
        approaches=["fourier", "chebyshev"],
        arangement="vertical")
plot_accuracy_vs_perf(
        df, methods=["sin", "square", "sawtooth"],
        metric="Bytes sent CP1", intervals=["(-9.42, 9.42)"],
        approaches=["fourier", "chebyshev"],
        arangement="vertical")

In [ ]:
""" E2E apps """

# Until Codon Jupyter is fixed: Read the data from files
dump_folder = "dump"
dump_files = [
    # "log_reg",
    # "mnist",
    # "dti",
    "gwas",
    "siren",
    "siren_cheby",
    # "siren_cheby_reduced",
    # "medmnist"
    ]
nbit_fs = [64]
cps = [0, 1]

df_data = {
    'Method': [],
    'Accuracy': [],
    'Loss': [],
    'Runtime': [],
    'Partitions count': [],
    'Truncations count': [],
    'Rounds': []
    }

df_gwas_betas = {
    'Method': [],
    'Betas': []
    }

df_siren_images = {
    'Method': [],
    'Images': []
    }

for cp in cps:
    df_data[f'Bytes sent CP{cp}'] = []
    df_data[f'Requests sent CP{cp}'] = []

for dump_file in dump_files:
    for nbit_f in nbit_fs:
        for cp in cps:
            try:
                with open(f"{dump_folder}/{dump_file}_{nbit_f}_CP{cp}.p", "rb") as f:
                    data = pickle.load(f)
                    for k, v in data.items():
                        if "gwas_plaintext" in k and "_betas" in k:
                            df_gwas_betas['Method'].append(f"{k}_{nbit_f}")
                            df_gwas_betas['Betas'].append(data[k])
                        if not k.endswith('_time'):
                            continue

                        k = k.replace('_time', '')
                        accuracy = np.array(data.get(f"{k}_accuracy", [-1])).mean()
                        loss = np.array(data.get(f"{k}_loss", [-1])).mean()
                        runtime = round(data.get(f"{k}_time", [-1])[0], 5)
                        bytes_sent = int(data.get(f"{k}_bytes_sent", [-1])[0])
                        send_requests = int(data.get(f"{k}_send_requests", [-1])[0])
                        partitions_count = int(data.get(f"{k}_partitions_count", [-1])[0])
                        truncations_count = int(data.get(f"{k}_truncations_count", [-1])[0])
                        rounds = int(data.get(f"{k}_rounds", [-1])[0])

                        if cp == 1:
                            df_data['Method'].append(f"{k}_{nbit_f}")
                            df_data['Accuracy'].append(accuracy)
                            df_data['Loss'].append(loss)
                            df_data['Runtime'].append(runtime)
                            df_data['Partitions count'].append(partitions_count)
                            df_data['Truncations count'].append(truncations_count)
                            df_data['Rounds'].append(rounds)

                            if dump_file == "gwas":
                                df_gwas_betas['Method'].append(f"{k}_betas_{nbit_f}")
                                df_gwas_betas['Betas'].append(data[f"{k}_betas"])
                            if dump_file.startswith("siren"):
                                df_siren_images['Method'].append(f"{k}_img_{nbit_f}")
                                df_siren_images['Images'].append(data[f"{k}_img"])
                        
                        df_data[f'Bytes sent CP{cp}'].append(bytes_sent)
                        df_data[f'Requests sent CP{cp}'].append(send_requests)
            except FileNotFoundError:
                print(f"Could not find {dump_folder}/{dump_file}_{nbit_f}_CP{cp}.p")

e2e_df = pd.DataFrame(df_data)
gwas_betas_df = pd.DataFrame(df_gwas_betas)
siren_images_df = pd.DataFrame(df_siren_images)

In [ ]:
siren_images_df

In [ ]:
# Average all plain, decor, and cheby methods for each application
# Average e2e_df per plain, decor, and chebyshev_degree in Method column
import re

df_tmp = e2e_df.copy()
# Extract base name like "chest_plain_0", "organ_chebyshev_20", ...
df_tmp['Base'] = df_tmp['Method'].str.replace(r'^siren_', '', regex=True).str.replace(r'_64$', '', regex=True)

# Numeric columns to average
numeric_cols = df_tmp.select_dtypes(include=[np.number]).columns.tolist()

# 1) Average per Base (i.e., per chest/oct/organ + approach+degree)
e2e_avg_by_base = df_tmp.groupby('Base')[numeric_cols].mean().reset_index()

# 2) Average per approach key (plain / decor / chebyshev_<deg>)
def approach_key(base):
    m = re.search(r'chebyshev_\d+', base)
    if m:
        return m.group(0)
    parts = base.split('_')
    # expected forms: chest_plain_0, chest_decor_0 -> return 'plain' or 'decor'
    return parts[1] if len(parts) > 1 else base

df_tmp['ApproachKey'] = df_tmp['Base'].apply(approach_key)
e2e_avg_by_approach = df_tmp.groupby('ApproachKey')[numeric_cols].mean().reset_index()

display(e2e_avg_by_base)
display(e2e_avg_by_approach)

In [ ]:
def get_img_by_method(method_name, df):
    row = df[df['Method'] == method_name]
    if not row.empty:
        return row.iloc[0]['Images']
    return None

In [ ]:
resolution = 64
approaches = [["chest_plain_0", "oct_plain_0", "organ_plain_0"],
              ["chest_decor_0", "oct_decor_0", "organ_decor_0"],
              ["chest_chebyshev_10", "oct_chebyshev_10", "organ_chebyshev_10"],
              ["chest_chebyshev_20", "oct_chebyshev_20", "organ_chebyshev_20"]]

# Transpose approaches so we iterate across modalities (chest/oct/organ) per method
approaches = [list(t) for t in zip(*approaches)]
approaches = [name for group in approaches for name in group]

for name in approaches:
    img = np.array(get_img_by_method(f"siren_{name}_img_64", siren_images_df))
    try:
        img = img.reshape(resolution, resolution)
    except:
        print(f"Could not reshape image for method {name}")
        continue
    
    # Compute the gradient of img
    gx, gy = np.gradient(img)
    grad_img = np.hypot(gx, gy)

    # Compute the divergence of the gradient
    gxx, _ = np.gradient(gx)
    _, gyy = np.gradient(gy)
    div_img = np.hypot(gxx, gyy)
    
    print(f"Method: {name}")    
    plt.imshow(img, cmap='gray')
    plt.axis('off')
    plt.show()

    plt.imshow(grad_img, cmap='inferno')
    plt.axis('off')
    plt.show()

    plt.imshow(div_img, cmap='turbo')
    plt.axis('off')
    plt.show()

In [ ]:
def plot_scatter_baseline_refs(
        x, ys, labels=None, title="Scatter Plot",
        xlabel="X", ylabel="Y",
        plot_labels=False):
    """
    Plots a scatter plot with a baseline X and multiple reference Ys.

    Args:
        x (list or np.ndarray): Baseline X values.
        ys (list of lists or np.ndarrays): List of reference Y values.
        labels (list of str, optional): Labels for each Y reference.
        title (str, optional): Plot title.
        xlabel (str, optional): Label for X axis.
        ylabel (str, optional): Label for Y axis.
    """
    plt.figure(figsize=(8, 6))
    
    # draw identity (ground truth) diagonal and set equal aspect & limits
    arrs = [np.asarray(x)] + [np.asarray(y) for y in ys if y is not None]
    if len(arrs) > 0 and all(a.size > 0 for a in arrs):
        concat = np.concatenate([a.ravel() for a in arrs])
        mn, mx = float(concat.min()), float(concat.max())
        pad = (mx - mn) * 0.02 if mx > mn else 1.0
        line_x = np.linspace(mn - pad, mx + pad, 100)
        plt.plot(line_x, line_x, '--', color='tab:green', linewidth=0.77, label='Ground truth')
        plt.xlim(mn - pad, mx + pad)
        plt.ylim(mn - pad, mx + pad)
        plt.gca().set_aspect('equal', adjustable='box')
    
    for idx, y in enumerate(ys):
        label = labels[idx] if labels and idx < len(labels) else f"Ref {idx+1}"
        plt.scatter(x, y, label=label, s=10)
        # Print mean absolute error between x and y
        mae_value = mae(x, y)
        print(f"{label} MAE: {mae_value:.2e}")

    if plot_labels:
        plt.title(title)
        plt.xlabel(xlabel)
        plt.ylabel(ylabel)
        plt.legend()
    
    plt.tick_params(axis='both', which='major', labelsize=19)
    plt.tick_params(axis='both', which='minor', labelsize=19)
    plt.show()

In [ ]:
def get_betas_by_method(method_name, df):
    row = df[df['Method'] == method_name]
    if not row.empty:
        return row.iloc[0]['Betas']
    return None

In [ ]:
for degree in [20, 30, 40, 50]:
    plot_scatter_baseline_refs(
        x=get_betas_by_method(f"gwas_plaintext_betas_64", gwas_betas_df),
        ys=[
            get_betas_by_method(f"gwas_decor_betas_64", gwas_betas_df),
            get_betas_by_method(f"gwas_chebyshev_{degree}_betas_64", gwas_betas_df),
        ],
        labels=["Decor", "Chebyshev", "Fourier"],
        title=f"GWAS Betas Comparison {degree}",
        xlabel="Plaintext Betas",
        ylabel="Secure Betas")

In [ ]:
e2e_df[e2e_df['Method'].str.contains('siren')].sort_values(by='Accuracy', ascending=False)

In [ ]:
e2e_df[e2e_df['Method'].str.contains('gwas')].sort_values(by='Accuracy', ascending=False)